In [3]:
# ==============================================================================
# CLUSTERING PIPELINE — SPLIT DATASET — HDBSCAN
# pour 30000 patients → GPU ok
# pour 120000 patients → basculer sur CPU (voir compute_distance_matrix commenté)
# ==============================================================================


# ==============================================================================
# 0. CONFIGURATION
# ==============================================================================

# ── Paths ──────────────────────────────────────────────────────────────────────
CSV_PATH   = "df_final_binaire_imputed.csv"
OUTPUT_DIR = "Results/Regular_clustering/Split_dataset"

# ── GPU ────────────────────────────────────────────────────────────────────────
GPU_DEVICE_ID = 1

# ── UMAP (réduction dimensionnelle avant HDBSCAN) ─────────────────────────────
UMAP_N_NEIGHBORS  = 30
UMAP_MIN_DIST     = 0.0
UMAP_N_COMPONENTS = 10
UMAP_RANDOM_STATE = 42

# ── UMAP / t-SNE (visualisation uniquement) ───────────────────────────────────
VIZ_N_NEIGHBORS = 30
VIZ_MIN_DIST    = 0.1
VIZ_TSNE_PERP   = 30
VIZ_TSNE_ITER   = 1500

# ── HDBSCAN ────────────────────────────────────────────────────────────────────
HDBSCAN_MIN_CLUSTER_SIZE = 500
HDBSCAN_MIN_SAMPLES      = 10
HDBSCAN_CLUSTER_METHOD   = "eom"

# ── Sweep ──────────────────────────────────────────────────────────────────────
SWEEP_VALUES = list(range(200, 2100, 100))

# ── Combined score weights ─────────────────────────────────────────────────────
W_SILHOUETTE = 0.4
W_STABILITY  = 0.3
W_OUTLIER    = 0.3

# ── Poids scenario_2 ──────────────────────────────────────────────────────────
WEIGHT_HOSP_S2 = 3.0

# ── Poids scenario_3 ──────────────────────────────────────────────────────────
WEIGHT_BIO_VEINOUS      = 1 / 29
WEIGHT_IMAGING_DETAILED = 1 / 12
WEIGHT_HOSP_S3          = 5.0

# ── Categorisation bins ────────────────────────────────────────────────────────
BIO_BINS    = [-1, 0, 1, 2, 10]
BIO_LABELS  = ["0", "1", "2", "3+"]
IMAG_BINS   = [-1, 0, 1, 2, 10]
IMAG_LABELS = ["0", "1", "2", "3+"]


# ==============================================================================
# 1. IMPORTS
# ==============================================================================

import os
import logging
import time

import gower
import hdbscan
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import numpy as np
import pandas as pd
import plotly.express as px
import seaborn as sns
import torch
import umap
import umap.umap_ as umap_reduce

from scipy.cluster.hierarchy import dendrogram, linkage
from sklearn.manifold import TSNE
from sklearn.metrics import pairwise_distances, silhouette_score
from sklearn.preprocessing import StandardScaler, MinMaxScaler

log = logging.getLogger(__name__)
logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s")

# ── GPU setup ──────────────────────────────────────────────────────────────────
torch.cuda.set_device(GPU_DEVICE_ID)
torch.cuda.set_per_process_memory_fraction(0.5, device=GPU_DEVICE_ID)
print(f"GPU : {torch.cuda.get_device_name(GPU_DEVICE_ID)}")
print(f"Mémoire disponible : {torch.cuda.get_device_properties(GPU_DEVICE_ID).total_memory / 1e9:.0f} GB")


# ==============================================================================
# 2. COLUMN DEFINITIONS
# ==============================================================================

IMAGING_COLS_BOOL = {
    "has_ultrasound", "has_ct_scan", "has_xray",
    "has_mri", "has_radio_interventional", "has_nuclear_medicine",
}

IMAGING_COLS_DETAILED = {
    "ultrasound_1", "ultrasound_2",
    "ct_scan_1", "ct_scan_2", "ct_scan_3",
    "xray_1", "xray_2", "xray_3",
    "mri_1", "mri_2",
    "radio_interventional_1",
    "nuclear_medicine_1",
}

BIO_VEINOUS = {
    "is_hemoglobine", "is_leucocytes", "is_formule_leuco",
    "is_urea", "is_creatinine", "is_sodium", "is_potassium",
    "is_platelets", "is_pt", "is_aptt", "is_calcium", "is_ck",
    "is_lactates", "is_troponine", "is_bnp", "is_ckmb", "is_ddimer",
    "is_crp", "is_pct", "is_alat", "is_asat", "is_bili_total",
    "is_lipase", "is_alp", "is_iron", "is_ferritin",
    "is_calcium_ionized", "is_aXa_aIIa", "is_fibrinogen",
}

BIO_EXAMS = {
    "has_blood_test", "has_culture",
    "has_lumbar_puncture", "has_blood_gas",
}

PROCEDURE_COLS   = {"had_ekg"}
DISPOSITION_COLS = {
    "hospitalization", "observation_unit", "inter_facility_transfer",
}

COLS_QUANTI = ["imaging_exam_count", "bio_exam_count"]

COLS_BINARY    = list(IMAGING_COLS_BOOL | BIO_EXAMS | PROCEDURE_COLS | BIO_VEINOUS | DISPOSITION_COLS)
COLS_MULTI_CAT = list(IMAGING_COLS_DETAILED)

SCENARIOS = {
    "scenario_2": list(
        IMAGING_COLS_BOOL | BIO_EXAMS | PROCEDURE_COLS | DISPOSITION_COLS
    ) + COLS_QUANTI,

    "scenario_3": list(
        IMAGING_COLS_BOOL | BIO_EXAMS | PROCEDURE_COLS | DISPOSITION_COLS
        | IMAGING_COLS_DETAILED | BIO_VEINOUS
    ) + COLS_QUANTI,
}


# ==============================================================================
# 3. DATA LOADING & PREPROCESSING
# ==============================================================================

def load_and_preprocess(csv_path: str = CSV_PATH) -> pd.DataFrame:
    """Load raw CSV and add bio/imaging ordinal category columns."""
    df = pd.read_csv(csv_path, low_memory=False)

    df["bio_exam_cat"] = pd.cut(
        df["bio_exam_count"], bins=BIO_BINS, labels=BIO_LABELS
    )
    df["imaging_exam_cat"] = pd.cut(
        df["imaging_exam_count"], bins=IMAG_BINS, labels=IMAG_LABELS
    )

    for col in ("bio_exam_count", "imaging_exam_count"):
        dist = (
            df[col].value_counts(dropna=False).sort_index()
            .rename_axis(col).reset_index(name="n")
        )
        dist["pct"] = (dist["n"] / dist["n"].sum() * 100).round(1)
        log.info(f"\n{col} distribution:\n{dist.to_string(index=False)}")

    return df


# ==============================================================================
# 4. DISTANCE MATRIX — GPU
# ==============================================================================

def compute_distance_matrix_gpu(
    df_sub:        pd.DataFrame,
    binary_cols:   list,
    cat_cols:      list,
    quanti_cols:   list,
    weight_hosp:   float = 1.0,
    weight_quanti: float = 1.0,
    device_id:     int   = GPU_DEVICE_ID,
) -> np.ndarray:
    """
    Pairwise distance matrix on GPU (PyTorch).
    Binary/cat → Hamming | Quanti → Manhattan StandardScaler-normalised.
    Returns float64 numpy array.
    """
    device = torch.device(f"cuda:{device_id}")
    n      = len(df_sub)
    D      = torch.zeros((n, n), device=device, dtype=torch.float32)

    for col in binary_cols:
        vals = torch.tensor(
            df_sub[col].values, dtype=torch.float32, device=device
        ).unsqueeze(1)
        d = torch.cdist(vals, vals, p=1)
        w = weight_hosp if col == "hospitalization" else 1.0
        D += d * w

    for col in cat_cols:
        codes = torch.tensor(
            df_sub[col].cat.codes.values, dtype=torch.float32, device=device
        ).unsqueeze(1)
        D += (codes != codes.T).float()

    if quanti_cols:
        q        = df_sub[quanti_cols].values.astype("float32")
        q_scaled = StandardScaler().fit_transform(q).astype("float32")
        q_tensor = torch.tensor(q_scaled, dtype=torch.float32, device=device)
        D       += torch.cdist(q_tensor, q_tensor, p=1) * weight_quanti

    D.fill_diagonal_(0)
    return D.cpu().numpy().astype(np.float64)


# def compute_distance_matrix(
#     df_sub:        pd.DataFrame,
#     binary_cols:   list,
#     cat_cols:      list,
#     quanti_cols:   list,
#     weight_hosp:   float = 1.0,
#     weight_quanti: float = 1.0,
# ) -> np.ndarray:
#     """CPU version — activate for 120k patients."""
#     n = len(df_sub)
#     D = np.zeros((n, n), dtype=np.float64)
#     for col in binary_cols:
#         vals = df_sub[col].values.reshape(-1, 1)
#         d    = pairwise_distances(vals, metric="hamming")
#         w    = weight_hosp if col == "hospitalization" else 1.0
#         D   += d * w
#     for col in cat_cols:
#         vals = df_sub[col].cat.codes.values.reshape(-1, 1)
#         D   += pairwise_distances(vals, metric="hamming")
#     if quanti_cols:
#         q        = df_sub[quanti_cols].values.astype("float64")
#         q_scaled = StandardScaler().fit_transform(q)
#         D       += pairwise_distances(q_scaled, metric="manhattan") * weight_quanti
#     np.fill_diagonal(D, 0)
#     return D


# ==============================================================================
# 5. CLUSTERING — run_hdbscan
# ==============================================================================

def run_hdbscan(
    df:               pd.DataFrame,
    scenario_name:    str,
    run_label:        str,
    distance_metric:  str,
    weight_hosp:      float = 1.0,
    weight_quanti:    float = 1.0,
    gower_weights:    dict  = None,
    min_cluster_size: int   = HDBSCAN_MIN_CLUSTER_SIZE,
    min_samples:      int   = HDBSCAN_MIN_SAMPLES,
):
    """Run HDBSCAN clustering. Returns df_sub, D, labels, clusterer."""

    sc_dir = os.path.join(OUTPUT_DIR, run_label)
    os.makedirs(sc_dir, exist_ok=True)

    cols   = [c for c in SCENARIOS[scenario_name] if c in df.columns]
    df_sub = df[cols].copy()

    if scenario_name == "scenario_3":
        essential = [c for c in cols if c not in IMAGING_COLS_DETAILED and c not in BIO_VEINOUS]
        df_sub    = df_sub.dropna(subset=essential)
    else:
        df_sub = df_sub.dropna()

    idx = df_sub.index

    quanti_cols = [c for c in COLS_QUANTI if c in df_sub.columns]
    binary_cols = [
        c for c in df_sub.columns
        if c not in quanti_cols and set(df_sub[c].dropna().unique()) <= {0, 1}
    ]
    cat_cols = [
        c for c in df_sub.columns
        if c not in binary_cols and c not in quanti_cols
    ]
    for col in cat_cols:
        df_sub[col] = df_sub[col].astype("category")

    log.info(
        f"[{run_label}] binary={len(binary_cols)} | "
        f"cat={len(cat_cols)} | quanti={len(quanti_cols)} | n={len(df_sub)}"
    )

    t0       = time.time()
    col_list = list(df_sub.columns)
    n_vars   = len(col_list)

    if distance_metric == "gower":
        df_sub = df_sub.copy()
        for col in df_sub.select_dtypes(include="integer").columns:
            df_sub[col] = df_sub[col].astype("float64")
        for col in df_sub.select_dtypes(include="category").columns:
            df_sub[col] = df_sub[col].astype("object")

        weights = np.ones(n_vars)
        if gower_weights:
            for col, w in gower_weights.items():
                if col in col_list:
                    weights[col_list.index(col)] = w
        weights = weights / weights.sum() * n_vars

        D              = gower.gower_matrix(df_sub, weight=weights).astype(np.float64)
        np.fill_diagonal(D, 0)
        fit_input      = D
        hdbscan_metric = "precomputed"

    elif distance_metric == "precomputed":
        D = compute_distance_matrix_gpu(
            df_sub,
            binary_cols   = binary_cols,
            cat_cols      = cat_cols,
            quanti_cols   = quanti_cols,
            weight_hosp   = weight_hosp,
            weight_quanti = weight_quanti,
        )
        # ── A ACTIVER POUR CPU (120k patients) ─────────────────────────────────
        # D = compute_distance_matrix(
        #     df_sub,
        #     binary_cols   = binary_cols,
        #     cat_cols      = cat_cols,
        #     quanti_cols   = quanti_cols,
        #     weight_hosp   = weight_hosp,
        #     weight_quanti = weight_quanti,
        # )
        fit_input      = D
        hdbscan_metric = "precomputed"

    else:
        raise ValueError(f"distance_metric must be 'gower' or 'precomputed', not '{distance_metric}'.")

    log.info(f"[{run_label}] Distance matrix : {time.time()-t0:.1f}s")

    if scenario_name == "scenario_3":
        t0      = time.time()
        reducer = umap_reduce.UMAP(
            n_neighbors  = UMAP_N_NEIGHBORS,
            min_dist     = UMAP_MIN_DIST,
            n_components = UMAP_N_COMPONENTS,
            metric       = "precomputed",
            random_state = UMAP_RANDOM_STATE,
        )
        emb = reducer.fit_transform(D)
        np.save(os.path.join(sc_dir, "umap_embedding.npy"), emb)
        log.info(f"[{run_label}] UMAP embedding shape={emb.shape} — {time.time()-t0:.1f}s")
        fit_input      = emb
        hdbscan_metric = "euclidean"

    t0 = time.time()
    clusterer = hdbscan.HDBSCAN(
        min_cluster_size         = min_cluster_size,
        min_samples              = min_samples,
        metric                   = hdbscan_metric,
        cluster_selection_method = HDBSCAN_CLUSTER_METHOD,
        gen_min_span_tree        = True,
    ).fit(fit_input)
    log.info(f"[{run_label}] HDBSCAN : {time.time()-t0:.1f}s")

    labels     = clusterer.labels_
    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
    n_noise    = (labels == -1).sum()
    log.info(
        f"[{run_label}] clusters={n_clusters} | "
        f"noise={n_noise} ({100*n_noise/len(labels):.1f}%)"
    )

    df_out            = df.loc[idx].copy()
    df_out["cluster"] = labels
    df_out.to_csv(os.path.join(sc_dir, f"clustering_mcs{min_cluster_size}.csv"), index=False)

    return df_sub, D, labels, clusterer


# ==============================================================================
# 6. SWEEP
# ==============================================================================

def run_hdbscan_sweep(
    D:            np.ndarray,
    sweep_values: list = SWEEP_VALUES,
    metric:       str  = "precomputed",
    min_samples:  int  = HDBSCAN_MIN_SAMPLES,
) -> pd.DataFrame:
    """Sweep over min_cluster_size and collect quality metrics."""
    np.fill_diagonal(D, 0)
    results = []

    for mcs in sweep_values:
        clusterer = hdbscan.HDBSCAN(
            min_cluster_size         = mcs,
            min_samples              = min_samples,
            metric                   = metric,
            cluster_selection_method = HDBSCAN_CLUSTER_METHOD,
        ).fit(D)

        labels          = clusterer.labels_
        unique_clusters = np.unique(labels[labels >= 0])
        n_clusters      = len(unique_clusters)

        sil = (
            silhouette_score(D, labels, metric=metric)
            if n_clusters >= 2 else np.nan
        )
        stability = (
            float(np.mean(clusterer.cluster_persistence_))
            if len(clusterer.cluster_persistence_) > 0 else np.nan
        )

        results.append({
            "min_cluster_size": mcs,
            "n_clusters":       n_clusters,
            "silhouette":       sil,
            "outlier_rate":     float(np.mean(labels == -1)),
            "stability":        stability,
            "labels":           labels,
            "probabilities":    clusterer.probabilities_,
            "cluster_sizes":    {c: int(np.sum(labels == c)) for c in unique_clusters},
        })

    return pd.DataFrame(results)


def compute_combined_score(
    df_sweep: pd.DataFrame,
    w_sil:    float = W_SILHOUETTE,
    w_stab:   float = W_STABILITY,
    w_out:    float = W_OUTLIER,
) -> pd.DataFrame:
    """Add combined_score column to sweep results."""
    df = df_sweep.copy()
    df["silhouette"] = df["silhouette"].fillna(0)
    df["stability"]  = df["stability"].fillna(0)

    def _minmax(s):
        mn, mx = s.min(), s.max()
        return (s - mn) / (mx - mn) if mx != mn else s * 0

    df["combined_score"] = (
        w_sil * _minmax(df["silhouette"])
        + w_stab * _minmax(df["stability"])
        - w_out * df["outlier_rate"]
    )
    return df


# ==============================================================================
# 7. VISUALISATION — SWEEP CURVES
# ==============================================================================

def plot_sweep_curves(df_sweep, title, out_dir, filename_prefix):
    os.makedirs(out_dir, exist_ok=True)
    x      = df_sweep["min_cluster_size"]
    y_sil  = df_sweep["silhouette"]
    y_stab = df_sweep["stability"]
    y_comb = df_sweep["combined_score"]

    fig, ax1 = plt.subplots(figsize=(10, 6))
    ax1.plot(x, y_sil, color="tab:blue", marker="o", label="Silhouette")
    ax1.set_xlabel("min_cluster_size")
    ax1.set_ylabel("Silhouette score", color="tab:blue")
    ax1.tick_params(axis="y", labelcolor="tab:blue")
    ax2 = ax1.twinx()
    ax2.plot(x, y_stab, color="tab:red", marker="s", label="Stability")
    ax2.set_ylabel("Mean stability", color="tab:red")
    ax2.tick_params(axis="y", labelcolor="tab:red")
    plt.title(title)
    fig.tight_layout()
    path1 = os.path.join(out_dir, f"{filename_prefix}_silhouette_stability.png")
    plt.savefig(path1, dpi=150)
    plt.close()
    log.info(f"Saved: {path1}")

    plt.figure(figsize=(10, 6))
    plt.plot(x, y_comb, color="tab:green", marker="d", linewidth=2)
    plt.xlabel("min_cluster_size")
    plt.ylabel("Combined score")
    plt.title("Combined score (silhouette + stability - outliers)")
    plt.grid(True)
    path2 = os.path.join(out_dir, f"{filename_prefix}_combined_score.png")
    plt.savefig(path2, dpi=150)
    plt.close()
    log.info(f"Saved: {path2}")


# ==============================================================================
# 8. VISUALISATION — CLUSTER PROFILES
# ==============================================================================

def plot_cluster_heatmap(df_sub, labels, title, out_dir, filename):
    os.makedirs(out_dir, exist_ok=True)
    df_prof            = df_sub.copy()
    df_prof["cluster"] = labels
    df_prof            = df_prof[df_prof["cluster"] != -1]

    numeric_cols = [
        c for c in df_prof.select_dtypes(include=["number"]).columns.tolist()
        if c != "cluster"
    ]
    profile = df_prof[numeric_cols + ["cluster"]].groupby("cluster").mean().round(3)

    n_cols = profile.shape[1]
    n_rows = profile.shape[0]

    plt.figure(figsize=(max(8, n_cols * 0.8), max(4, n_rows * 0.6)))
    sns.heatmap(
        profile, annot=True, cmap="YlOrRd", fmt=".2f",
        annot_kws={"size": max(6, min(10, 80 // n_cols))},
    )
    plt.title(title)
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    path = os.path.join(out_dir, filename)
    plt.savefig(path, dpi=150, bbox_inches="tight")
    plt.close()
    log.info(f"Saved: {path}")
    return profile


def plot_cluster_dendrogram(profile, title, out_dir, filename):
    os.makedirs(out_dir, exist_ok=True)
    Z               = linkage(profile.values, method="ward")
    variables       = profile.columns.tolist()
    cluster_vectors = {i: profile.iloc[i].values for i in range(len(profile))}

    plt.figure(figsize=(10, 5))
    dendrogram(Z, labels=profile.index.astype(str))
    plt.gca().grid(False)
    plt.title(title)

    for i, (c1, c2, dist, _) in enumerate(Z):
        c1, c2  = int(c1), int(c2)
        v1, v2  = cluster_vectors[c1], cluster_vectors[c2]
        top_var = variables[np.argmax(np.abs(v1 - v2))]
        cluster_vectors[len(cluster_vectors)] = (v1 + v2) / 2
        plt.text(i + 1, dist, top_var, rotation=45, fontsize=8, va="bottom", ha="center")

    plt.tight_layout()
    path = os.path.join(out_dir, filename)
    plt.savefig(path, dpi=150)
    plt.close()
    log.info(f"Saved: {path}")


# ==============================================================================
# 9. VISUALISATION — UMAP & t-SNE
# ==============================================================================

def build_cluster_palette(labels: np.ndarray) -> dict:
    """Build consistent {cluster_id: color} dict. Outliers → lightgrey."""
    unique_clusters = sorted([c for c in np.unique(labels) if c != -1])
    n_clusters      = len(unique_clusters)
    palette         = sns.color_palette("tab20", n_clusters) if n_clusters <= 20 \
                      else sns.color_palette("hsv", n_clusters)
    color_map       = {c: palette[i] for i, c in enumerate(unique_clusters)}
    color_map[-1]   = "lightgrey"
    return color_map


def _to_hex(color):
    """Convert matplotlib color to hex for plotly."""
    if color == "lightgrey":
        return "#d3d3d3"
    return mcolors.to_hex(color)


def plot_umap_2d(X, labels, title, out_dir, filename, metric="precomputed"):
    os.makedirs(out_dir, exist_ok=True)
    emb             = umap.UMAP(
        n_components=2, n_neighbors=VIZ_N_NEIGHBORS,
        min_dist=VIZ_MIN_DIST, metric=metric,
    ).fit_transform(X)
    color_map       = build_cluster_palette(labels)
    unique_clusters = sorted([c for c in np.unique(labels) if c != -1])

    plt.figure(figsize=(9, 7))
    mask_noise = labels == -1
    if mask_noise.any():
        plt.scatter(emb[mask_noise, 0], emb[mask_noise, 1],
                    c="lightgrey", s=8, linewidth=0,
                    label="Outliers (-1)", zorder=1, alpha=0.5)
    for c in unique_clusters:
        mask = labels == c
        plt.scatter(emb[mask, 0], emb[mask, 1],
                    color=color_map[c], s=10, linewidth=0,
                    label=f"C{c}", zorder=2, alpha=0.8)
    plt.title(title)
    plt.legend(title="Cluster", bbox_to_anchor=(1.05, 1),
               loc="upper left", markerscale=2, fontsize=8)
    plt.tight_layout()
    path = os.path.join(out_dir, filename)
    plt.savefig(path, dpi=150)
    plt.close()
    log.info(f"Saved: {path}")
    return color_map


def plot_umap_3d_html(X, labels, title, out_dir, filename_html,
                      metric="precomputed", color_map: dict = None):
    os.makedirs(out_dir, exist_ok=True)
    emb = umap.UMAP(
        n_components=3, n_neighbors=VIZ_N_NEIGHBORS,
        min_dist=VIZ_MIN_DIST, metric=metric,
    ).fit_transform(X)
    if color_map is None:
        color_map = build_cluster_palette(labels)
    unique_clusters = sorted([c for c in np.unique(labels) if c != -1])

    df_plot = pd.DataFrame({
        "x": emb[:, 0], "y": emb[:, 1], "z": emb[:, 2],
        "label": [str(l) for l in labels],
        "color": [_to_hex(color_map[l]) for l in labels],
    })
    df_plot["order"] = df_plot["label"].apply(lambda x: -1 if x == "-1" else int(x))
    df_plot = df_plot.sort_values("order")

    fig = px.scatter_3d(
        df_plot, x="x", y="y", z="z", color="label",
        color_discrete_map={str(c): _to_hex(color_map[c])
                            for c in list(unique_clusters) + [-1]},
        title=title, opacity=0.8,
        category_orders={"label": ["-1"] + [str(c) for c in unique_clusters]},
    )
    fig.update_traces(marker=dict(size=3))
    fig.update_layout(legend_title_text="Cluster")
    path = os.path.join(out_dir, filename_html)
    fig.write_html(path, include_plotlyjs="cdn")
    log.info(f"Saved: {path}")


def plot_tsne_2d(X, labels, title, out_dir, filename,
                 metric="precomputed", color_map: dict = None):
    os.makedirs(out_dir, exist_ok=True)
    init = umap.UMAP(
        n_components=2, n_neighbors=VIZ_N_NEIGHBORS,
        min_dist=VIZ_MIN_DIST, metric=metric,
    ).fit_transform(X)
    emb = TSNE(
        n_components=2, perplexity=VIZ_TSNE_PERP,
        learning_rate="auto", init=init,
        metric=metric, max_iter=VIZ_TSNE_ITER,
    ).fit_transform(X)
    if color_map is None:
        color_map = build_cluster_palette(labels)
    unique_clusters = sorted([c for c in np.unique(labels) if c != -1])

    plt.figure(figsize=(9, 7))
    mask_noise = labels == -1
    if mask_noise.any():
        plt.scatter(emb[mask_noise, 0], emb[mask_noise, 1],
                    c="lightgrey", s=8, linewidth=0,
                    label="Outliers (-1)", zorder=1, alpha=0.5)
    for c in unique_clusters:
        mask = labels == c
        plt.scatter(emb[mask, 0], emb[mask, 1],
                    color=color_map[c], s=10, linewidth=0,
                    label=f"C{c}", zorder=2, alpha=0.8)
    plt.title(title)
    plt.legend(title="Cluster", bbox_to_anchor=(1.05, 1),
               loc="upper left", markerscale=2, fontsize=8)
    plt.tight_layout()
    path = os.path.join(out_dir, filename)
    plt.savefig(path, dpi=150)
    plt.close()
    log.info(f"Saved: {path}")


def plot_tsne_3d_html(X, labels, title, out_dir, filename_html,
                      metric="precomputed", color_map: dict = None):
    os.makedirs(out_dir, exist_ok=True)
    init = umap.UMAP(
        n_components=3, n_neighbors=VIZ_N_NEIGHBORS,
        min_dist=VIZ_MIN_DIST, metric=metric,
    ).fit_transform(X)
    emb = TSNE(
        n_components=3, perplexity=VIZ_TSNE_PERP,
        learning_rate="auto", init=init,
        metric=metric, max_iter=VIZ_TSNE_ITER,
    ).fit_transform(X)
    if color_map is None:
        color_map = build_cluster_palette(labels)
    unique_clusters = sorted([c for c in np.unique(labels) if c != -1])

    df_plot = pd.DataFrame({
        "x": emb[:, 0], "y": emb[:, 1], "z": emb[:, 2],
        "label": [str(l) for l in labels],
    })
    fig = px.scatter_3d(
        df_plot, x="x", y="y", z="z", color="label",
        color_discrete_map={str(c): _to_hex(color_map[c])
                            for c in list(unique_clusters) + [-1]},
        title=title, opacity=0.8,
        category_orders={"label": ["-1"] + [str(c) for c in unique_clusters]},
    )
    fig.update_traces(marker=dict(size=3))
    fig.update_layout(legend_title_text="Cluster")
    path = os.path.join(out_dir, filename_html)
    fig.write_html(path, include_plotlyjs="cdn")
    log.info(f"Saved: {path}")


# ==============================================================================
# 10. OUTLIERS INTERNES
# ==============================================================================

def describe_outliers_internal(
    df_sub:    pd.DataFrame,
    labels:    np.ndarray,
    run_label: str,
    out_dir:   str,
):
    """Compare outliers vs clustered on clustering variables."""
    os.makedirs(out_dir, exist_ok=True)

    df_work             = df_sub.copy()
    df_work["cluster"]  = labels
    df_noise            = df_work[df_work["cluster"] == -1]
    df_clustered        = df_work[df_work["cluster"] != -1]
    n_noise             = len(df_noise)
    n_total             = len(df_work)

    log.info(f"[{run_label}] Outliers : {n_noise} / {n_total} ({100*n_noise/n_total:.1f}%)")

    if n_noise == 0:
        log.info("No outliers.")
        return

    numeric_cols = [c for c in df_sub.select_dtypes(include="number").columns]

    compare         = pd.DataFrame({
        "outliers":  df_noise[numeric_cols].mean().round(3),
        "clustered": df_clustered[numeric_cols].mean().round(3),
    })
    compare["diff"] = (compare["outliers"] - compare["clustered"]).round(3)
    compare         = compare.sort_values("diff", key=abs, ascending=False)

    fig, ax = plt.subplots(figsize=(max(8, len(numeric_cols) * 0.8), 4))
    sns.heatmap(compare[["outliers", "clustered"]].T,
                annot=True, fmt=".2f", cmap="YlOrRd",
                ax=ax, annot_kws={"size": 8})
    ax.set_title(f"Outliers vs clustered — {run_label}")
    plt.tight_layout()
    path = os.path.join(out_dir, "outliers_internal_heatmap.png")
    plt.savefig(path, dpi=150, bbox_inches="tight")
    plt.close()
    log.info(f"Saved: {path}")

    fig, ax = plt.subplots(figsize=(max(8, len(numeric_cols) * 0.8), 5))
    colors  = ["tab:red" if v > 0 else "tab:blue" for v in compare["diff"]]
    ax.bar(compare.index, compare["diff"], color=colors, alpha=0.8)
    ax.axhline(0, color="black", linewidth=0.8)
    ax.set_ylabel("Difference (outliers - clustered)")
    ax.set_title(f"Outliers vs clustered diff — {run_label}")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    path = os.path.join(out_dir, "outliers_internal_diff.png")
    plt.savefig(path, dpi=150, bbox_inches="tight")
    plt.close()
    log.info(f"Saved: {path}")

    compare.to_csv(os.path.join(out_dir, "outliers_internal_compare.csv"))
    log.info(f"[{run_label}] Internal outlier description complete.")


# ==============================================================================
# 11. POSTPROCESSING
# ==============================================================================

def run_postprocessing(df_sub, D, labels, run_label, out_dir):
    """Sweep + heatmap + dendrogram + UMAP + t-SNE + outliers internal."""
    t_start = time.time()

    # Sweep
    df_sweep = run_hdbscan_sweep(D, metric="precomputed")
    df_sweep = compute_combined_score(df_sweep)
    df_sweep.drop(columns=["labels", "probabilities", "cluster_sizes"]) \
            .to_csv(os.path.join(out_dir, "sweep.csv"), index=False)
    plot_sweep_curves(df_sweep, title=f"Sweep — {run_label}",
                      out_dir=out_dir, filename_prefix=run_label)

    # Heatmap + dendrogram
    profile = plot_cluster_heatmap(
        df_sub, labels,
        title=f"Cluster profiles — {run_label}",
        out_dir=out_dir, filename="heatmap.png",
    )
    if len(profile) >= 2:
        plot_cluster_dendrogram(
            profile, title=f"Dendrogram — {run_label}",
            out_dir=out_dir, filename="dendrogram.png",
        )

    # UMAP 2D → generates palette
    color_map = plot_umap_2d(
        D, labels, title=f"UMAP 2D — {run_label}",
        out_dir=out_dir, filename="umap2d.png",
    )

    # UMAP 3D
    plot_umap_3d_html(
        D, labels, title=f"UMAP 3D — {run_label}",
        out_dir=out_dir, filename_html="umap3d.html",
        color_map=color_map,
    )

    # t-SNE 2D
    plot_tsne_2d(
        D, labels, title=f"t-SNE 2D — {run_label}",
        out_dir=out_dir, filename="tsne2d.png",
        color_map=color_map,
    )

    # t-SNE 3D
    plot_tsne_3d_html(
        D, labels, title=f"t-SNE 3D — {run_label}",
        out_dir=out_dir, filename_html="tsne3d.html",
        color_map=color_map,
    )

    # Outliers internes
    describe_outliers_internal(
        df_sub=df_sub, labels=labels,
        run_label=run_label,
        out_dir=os.path.join(out_dir, "outliers_internal"),
    )

    log.info(f"[{run_label}] Postprocessing done — {time.time()-t_start:.1f}s")


# ==============================================================================
# 12. RERUN WITHOUT SWEEP
# ==============================================================================

def rerun_postprocessing_no_sweep(
    df:               pd.DataFrame,
    scenario_name:    str   = "scenario_2",
    run_label:        str   = "s2_hospitalized",
    distance_metric:  str   = "precomputed",
    min_cluster_size: int   = 500,
    weight_hosp:      float = 1.0,
    weight_quanti:    float = 1.0,
    gower_weights:    dict  = None,
):
    """Re-run clustering with a specific mcs, regenerate all visuals except sweep."""
    out_dir = os.path.join(OUTPUT_DIR, run_label)
    os.makedirs(out_dir, exist_ok=True)

    df_sub, D, labels, clusterer = run_hdbscan(
        df,
        scenario_name    = scenario_name,
        run_label        = run_label,
        distance_metric  = distance_metric,
        weight_hosp      = weight_hosp,
        weight_quanti    = weight_quanti,
        gower_weights    = gower_weights,
        min_cluster_size = min_cluster_size,
    )

    profile = plot_cluster_heatmap(
        df_sub, labels,
        title    = f"Cluster profiles — {run_label} mcs={min_cluster_size}",
        out_dir  = out_dir,
        filename = f"heatmap_mcs{min_cluster_size}.png",
    )
    if len(profile) >= 2:
        plot_cluster_dendrogram(
            profile,
            title    = f"Dendrogram — {run_label} mcs={min_cluster_size}",
            out_dir  = out_dir,
            filename = f"dendrogram_mcs{min_cluster_size}.png",
        )

    color_map = plot_umap_2d(
        D, labels,
        title    = f"UMAP 2D — {run_label} mcs={min_cluster_size}",
        out_dir  = out_dir,
        filename = f"umap2d_mcs{min_cluster_size}.png",
    )
    plot_umap_3d_html(
        D, labels,
        title         = f"UMAP 3D — {run_label} mcs={min_cluster_size}",
        out_dir       = out_dir,
        filename_html = f"umap3d_mcs{min_cluster_size}.html",
        color_map     = color_map,
    )
    plot_tsne_2d(
        D, labels,
        title     = f"t-SNE 2D — {run_label} mcs={min_cluster_size}",
        out_dir   = out_dir,
        filename  = f"tsne2d_mcs{min_cluster_size}.png",
        color_map = color_map,
    )
    plot_tsne_3d_html(
        D, labels,
        title         = f"t-SNE 3D — {run_label} mcs={min_cluster_size}",
        out_dir       = out_dir,
        filename_html = f"tsne3d_mcs{min_cluster_size}.html",
        color_map     = color_map,
    )
    describe_outliers_internal(
        df_sub    = df_sub,
        labels    = labels,
        run_label = f"{run_label}_mcs{min_cluster_size}",
        out_dir   = os.path.join(out_dir, f"outliers_internal_mcs{min_cluster_size}"),
    )

    log.info(f"[{run_label}] ✅ Rerun mcs={min_cluster_size} complete.")
    return df_sub, D, labels, clusterer


# ==============================================================================
# 13. SPLIT PIPELINE
# ==============================================================================

def run_split_pipeline(df: pd.DataFrame):
    """
    Split dataset into hospitalized / discharged,
    save both subsets, then run scenario_2 no weights on each.
    """

    # ── Split ──────────────────────────────────────────────────────────────────
    df_hosp       = df[df["hospitalization"] == 1].copy().reset_index(drop=True)
    df_discharged = df[df["hospitalization"] == 0].copy().reset_index(drop=True)

    log.info(f"Split → hospitalized : {len(df_hosp)} | discharged : {len(df_discharged)}")

    # ── Save ───────────────────────────────────────────────────────────────────
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    path_hosp       = os.path.join(OUTPUT_DIR, "df_hospitalized.csv")
    path_discharged = os.path.join(OUTPUT_DIR, "df_discharged.csv")
    df_hosp.to_csv(path_hosp, index=False)
    df_discharged.to_csv(path_discharged, index=False)
    log.info(f"Saved: {path_hosp}")
    log.info(f"Saved: {path_discharged}")

    # ── Run scenario_2 no weights on each subset ───────────────────────────────
    runs = [
        dict(df=df_hosp,       run_label="s2_hospitalized"),
        dict(df=df_discharged, run_label="s2_discharged"),
    ]

    for run in runs:
        t_run     = time.time()
        run_label = run["run_label"]
        out_dir   = os.path.join(OUTPUT_DIR, run_label)

        log.info(f"\n{'='*60}")
        log.info(f"Run : {run_label} | n={len(run['df'])}")
        log.info(f"{'='*60}")

        df_sub, D, labels, clusterer = run_hdbscan(
            run["df"],
            scenario_name    = "scenario_2",
            run_label        = run_label,
            distance_metric  = "precomputed",
            weight_hosp      = 1.0,
            weight_quanti    = 1.0,
            gower_weights    = None,
            min_cluster_size = 500,
        )

        run_postprocessing(df_sub, D, labels, run_label, out_dir)
        log.info(f"[{run_label}] ✅ Run complet : {time.time()-t_run:.1f}s")


# ==============================================================================
# 14. ENTRY POINT
# ==============================================================================

if __name__ == "__main__":
    df = load_and_preprocess(CSV_PATH)
    run_split_pipeline(df)

GPU : NVIDIA A100-SXM4-80GB
Mémoire disponible : 85 GB


INFO | 
bio_exam_count distribution:
 bio_exam_count     n  pct
              0 12522 42.0
              1 12030 40.3
              2  4182 14.0
              3  1062  3.6
              4    43  0.1
INFO | 
imaging_exam_count distribution:
 imaging_exam_count     n  pct
                  0 14541 48.7
                  1 12968 43.5
                  2  2152  7.2
                  3   162  0.5
                  4    16  0.1
INFO | Split → hospitalized : 3757 | discharged : 26082
INFO | Saved: Results/Regular_clustering/Split_dataset/df_hospitalized.csv
INFO | Saved: Results/Regular_clustering/Split_dataset/df_discharged.csv
INFO | 
INFO | Run : s2_hospitalized | n=3757
INFO | ============================================================
INFO | [s2_hospitalized] binary=14 | cat=0 | quanti=2 | n=3757
INFO | [s2_hospitalized] Distance matrix : 0.5s
INFO | [s2_hospitalized] HDBSCAN : 1.2s
INFO | [s2_hospitalized] clusters=3 | noise=383 (10.2%)
INFO | Saved: Results/Regular_clustering/Split_da

## Analyse comparative des clusterings splitté (hospitalisés vs dischargés)

### Résultats du sweep

**Dischargés (n ≈ 21 000)**
- Stabilité = 1.0 pour tous les mcs
- Silhouette élevé (0.82 à mcs=200, 0.75 à mcs=500)
- Taux d'outliers très bas (1-5%)
- → Structure claire, stable et reproductible
- → mcs optimal retenu : **500 → 15 clusters**

**Hospitalisés (n ≈ 9 000)**
- Stabilité = 1.0 uniquement à mcs=200 (12 clusters)
- Silhouette bas (0.36) et chute rapide
- À partir de mcs=1500 → 0 clusters, tout en bruit
- → Structure très faible et instable

---

### Interprétation clinique

Les patients dischargés présentent des **profils de consommation de soins homogènes et distincts** — certains types de bilans sont reproductibles et prévisibles.

Les patients hospitalisés sont au contraire **très hétérogènes** : la décision d'hospitalisation dépend du diagnostic, de la gravité et du contexte social, facteurs qui ne sont pas capturés par les variables de consommation de soins seules. Ce n'est pas un manque de données — c'est une réalité clinique.

---

### Implications pour la prédiction au triage

- ✅ Il est possible de prédire le **profil de consommation de soins** d'un patient à l'arrivée
- ❌ Il n'est **pas possible** de prédire l'hospitalisation uniquement depuis les patterns de consommation, car les hospitalisés ne forment pas de clusters stables

---

### Décision méthodologique

Le clustering splitté est **abandonné** au profit du **clustering global (s2_noweights)** pour trois raisons :

1. Les hospitalisés ne forment pas de clusters stables → résultats peu fiables
2. L'objectif est la prédiction au triage → l'issue (hospitalisation) n'est pas encore connue à ce stade
3. Le clustering global est plus riche : les hospitalisés dispersés dans 4-5 clusters avec des taux de 25-60% constituent une information clinique en soi

La description des clusters globaux permettra d'identifier quels profils de consommation sont associés à un risque d'hospitalisation élevé, sans imposer artificiellement une séparation.

---

### Conclusion

> *"Les patients hospitalisés ne forment pas de profils de consommation de soins homogènes, suggérant que la décision d'hospitalisation dépend de facteurs cliniques au-delà de la consommation de soins mesurée aux urgences."*

## Caractérisation des outliers HDBSCAN

### Observation

Les patients classés en bruit (-1) par HDBSCAN sont systématiquement ceux ayant reçu le plus de soins, aussi bien dans la population des dischargés que des hospitalisés.

### Explication méthodologique

HDBSCAN classe un patient en bruit lorsqu'il ne se trouve pas dans une région suffisamment dense de l'espace de distances. Les patients avec un bilan intensif (scanner + échographie + biologie complète + EKG) présentent des profils très atypiques, sont peu nombreux, et sont donc isolés dans l'espace — ils n'ont pas assez de voisins similaires pour former un cluster.

### Interprétation clinique

La majorité des patients présente un bilan léger à modéré, formant des régions denses → clusters stables. Les patients les plus consommateurs de soins sont au contraire très rares et chacun présente un profil unique → outliers.

Cela reflète une réalité clinique connue : **les cas complexes sont tous différents**, tandis que les bilans standards sont reproductibles et prévisibles.

### Conclusion

> *"Les patients outliers présentent une consommation de soins significativement plus élevée que les patients clusterisés, suggérant que les profils de consommation intensive sont trop hétérogènes et trop rares pour former des groupes distincts. Ce résultat est cohérent avec la réalité clinique des urgences, où les cas complexes constituent des présentations uniques difficiles à typifier."*

In [5]:
# ==============================================================================
# CALL FOR SPECIFIC MCS AND SPECIFIC RUN POST SWEEP
# ==============================================================================



df = load_and_preprocess(CSV_PATH)

# ── Hospitalized — mcs = 1000 ──────────────────────────────────────────────────
df_hosp = df[df["hospitalization"] == 1].copy().reset_index(drop=True)

df_sub_h, D_h, labels_h, clusterer_h = rerun_postprocessing_no_sweep(
    df_hosp,
    scenario_name    = "scenario_2",
    run_label        = "s2_hospitalized",
    distance_metric  = "precomputed",
    min_cluster_size = 1000,
)

# ── Discharged — mcs = 1800 ───────────────────────────────────────────────────
df_discharged = df[df["hospitalization"] == 0].copy().reset_index(drop=True)

df_sub_d, D_d, labels_d, clusterer_d = rerun_postprocessing_no_sweep(
    df_discharged,
    scenario_name    = "scenario_2",
    run_label        = "s2_discharged",
    distance_metric  = "precomputed",
    min_cluster_size = 1800,
)

INFO | 
bio_exam_count distribution:
 bio_exam_count     n  pct
              0 12522 42.0
              1 12030 40.3
              2  4182 14.0
              3  1062  3.6
              4    43  0.1
INFO | 
imaging_exam_count distribution:
 imaging_exam_count     n  pct
                  0 14541 48.7
                  1 12968 43.5
                  2  2152  7.2
                  3   162  0.5
                  4    16  0.1
INFO | [s2_hospitalized] binary=14 | cat=0 | quanti=2 | n=3757
INFO | [s2_hospitalized] Distance matrix : 0.5s
INFO | [s2_hospitalized] HDBSCAN : 1.3s
INFO | [s2_hospitalized] clusters=2 | noise=28 (0.7%)
INFO | Saved: Results/Regular_clustering/Split_dataset/s2_hospitalized/heatmap_mcs1000.png
INFO | Saved: Results/Regular_clustering/Split_dataset/s2_hospitalized/dendrogram_mcs1000.png
/home/nadia/pycharm_project_nad/.venv/lib/python3.12/site-packages/umap/umap_.py:1865: UserWarning:

using precomputed metric; inverse_transform will be unavailable

INFO | Saved: Resu